In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [4]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [ ]:
# driver.quit()

### Investigate Missing Values

In [4]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [7]:
portal_df_test = pd.read_csv("data/portal_2025_transfers_1218_run5.csv")

In [8]:
portal_df_test.isna().sum()

id_247                     0
name                       0
pos_247                    0
hs_name                    5
hs_city                    0
hs_state                   0
transfer_rating            0
transfer_year              0
transfer_ovr_rank         88
transfer_pos_rank         35
transfer_stars             0
transfer_origin            5
transfer_destination      68
hs_class                   1
hs_rating_247           1012
hs_pos                     0
composite_rating        1121
composite_natl_rank     1123
composite_pos_rank      1123
source_hs_url              0
hs_stars                1012
source_player_url          0
transfer_status         2862
dtype: int64

In [13]:
# display(portal_df_test[portal_df_test['transfer_destination'].isna()].head())
# print(portal_df_test[portal_df_test['transfer_destination'].isna()].iloc[-2]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_destination'].isna()].iloc[-2])
# # 46137152 dropped out

# display(portal_df_test[portal_df_test['hs_name'].isna()].head(11))
# print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2]['source_player_url'])
# print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2])
# # https://247sports.com/player/louis-brown-iv-46111905/college-310862 is a problem
# # https://247sports.com/player/easton-messer-46103582/college-286927/

# display(portal_df_test[portal_df_test['transfer_rating'].isna()].head(11))
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2])
# fine

# display(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].head(11))
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4])
# fine


# display(portal_df_test[portal_df_test['pos_247'].isna()].head(11))
# print(portal_df_test[portal_df_test['pos_247'].isna()].iloc[-3]['source_player_url'])
# print(portal_df_test[portal_df_test['pos_247'].isna()].iloc[-3])

In [14]:
# from src.utils.tests import test_timeline
# test_timeline(driver,
#               'https://247sports.com/player/jackson-ford-46132876/college-318734')

In [15]:
# from src.scraper.player_scraper import scrape_player

# p = scrape_player(driver,
#                    'https://247sports.com/player/dylan-gooden-46116182/college-298901',)
# p

In [16]:
trouble_links = ['https://247sports.com/player/jackson-ford-46132876/college-318734',
 'https://247sports.com/player/aidan-glover-46131125/college-311282',
 'https://247sports.com/player/jj-harrell-46136887/college-308341',
 'https://247sports.com/player/cayman-spaulding-46154253/college-326362',
 'https://247sports.com/player/greg-johnson-ii-46117160/college-336698']

In [17]:
display(portal_df_test[portal_df_test['transfer_origin'].isna()][
    ['source_player_url', 'transfer_destination', 'transfer_origin', 'transfer_status']])

,source_player_url,transfer_destination,transfer_origin,transfer_status
2997,https://247sports.com/player/jackson-ford-4613...,NaN,NaN,none
3001,https://247sports.com/player/aidan-glover-4613...,NaN,NaN,none
3002,https://247sports.com/player/jj-harrell-461368...,NaN,NaN,none
3004,https://247sports.com/player/cayman-spaulding-...,NaN,NaN,none
3006,https://247sports.com/player/greg-johnson-ii-4...,NaN,NaN,none


In [18]:
display(portal_df_test[portal_df_test['transfer_origin'].isna()]['source_player_url'].tolist())

['https://247sports.com/player/jackson-ford-46132876/college-318734',
 'https://247sports.com/player/aidan-glover-46131125/college-311282',
 'https://247sports.com/player/jj-harrell-46136887/college-308341',
 'https://247sports.com/player/cayman-spaulding-46154253/college-326362',
 'https://247sports.com/player/greg-johnson-ii-46117160/college-336698']

In [20]:
portal_df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3008 entries, 0 to 3007
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_247                3008 non-null   int64  
 1   name                  3008 non-null   object 
 2   pos_247               3008 non-null   object 
 3   hs_name               3003 non-null   object 
 4   hs_city               3008 non-null   object 
 5   hs_state              3008 non-null   object 
 6   transfer_rating       3008 non-null   int64  
 7   transfer_year         3008 non-null   int64  
 8   transfer_ovr_rank     2920 non-null   float64
 9   transfer_pos_rank     2973 non-null   float64
 10  transfer_stars        3008 non-null   int64  
 11  transfer_origin       3003 non-null   object 
 12  transfer_destination  2940 non-null   object 
 13  hs_class              3007 non-null   float64
 14  hs_rating_247         1996 non-null   float64
 15  hs_pos               

In [ ]:
# --- What changed (high-level) ---
# 1) Added a GATE so only attempt fuzzy match for rows that actually need portal data:
#    - default: only fill if "Transfer Year" is blank (safe for reruns)
#    - optional: also require Snaps >= 100 if you have a "Snaps" column
# 2) Enforced ONE-TO-ONE matching: each 247 portal player (id_247) can only be assigned once.
# 3) Raised fuzzy threshold to 95 and switched scorer to WRatio (more robust).
# 4) Added quick duplicate diagnostics for portal_df by normalized name.

import re
import pandas as pd
from rapidfuzz import process, fuzz
from openpyxl import load_workbook

# ---------- helpers ----------
def norm_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b\.?", "", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_portal_lookup(portal_df: pd.DataFrame):
    pdf = portal_df.copy()
    pdf["name_norm"] = pdf["name"].map(norm_name)

    # keep "best" row per normalized name (highest transfer_rating, then hs_rating_247)
    pdf["_sort_transfer"] = pdf["transfer_rating"].fillna(-1)
    pdf["_sort_hs"] = pdf.get("hs_rating_247", pd.Series([None]*len(pdf))).fillna(-1)
    pdf = pdf.sort_values(["name_norm", "_sort_transfer", "_sort_hs"], ascending=[True, False, False])

    exact = dict(zip(pdf["name_norm"], pdf.index))
    choices = pdf["name_norm"].tolist()
    return pdf, exact, choices

def portal_to_fu_fields(row: pd.Series) -> dict:
    hs_city_state = None
    if pd.notna(row.get("hs_city")) and pd.notna(row.get("hs_state")):
        hs_city_state = f"{row.get('hs_city')}, {row.get('hs_state')}"
    hs_combo = None
    if pd.notna(row.get("hs_name")) and hs_city_state:
        hs_combo = f"{row.get('hs_name')} ({hs_city_state})"
    elif pd.notna(row.get("hs_name")):
        hs_combo = str(row.get("hs_name"))

    return {
        "247/On3 Position": row.get("pos_247"),
        "High School, City, State": hs_combo,
        "247 'Exp' (aka High School Class)": int(row["hs_class"]) if pd.notna(row.get("hs_class")) else None,
        "H/S Stars": int(row["hs_stars"]) if pd.notna(row.get("hs_stars")) else None,
        "H/S Rating": row.get("hs_rating_247"),
        "H/S National Rank": row.get("composite_natl_rank"),
        "H/S Position Rank": row.get("composite_pos_rank"),
        "Transfer Year": int(row["transfer_year"]) if pd.notna(row.get("transfer_year")) else None,
        "Transfer Origin": row.get("transfer_origin"),
        "Origin P4 / G5 / Non-FBS": None,
        "Transfer Destination": row.get("transfer_destination"),
        "Destination P4 / G5 / Non-FBS": None,
        "Transfer Stars": int(row["transfer_stars"]) if pd.notna(row.get("transfer_stars")) else None,
        "Transfer Rating": row.get("transfer_rating"),
        "Transfer Overall Rank": row.get("transfer_ovr_rank"),
        "Transfer Position Rank": row.get("transfer_pos_rank"),
    }

def fuzzy_match_one(name_raw: str, exact_map, choices, score_cutoff=95):
    n = norm_name(name_raw)
    if not n:
        return None, 0
    if n in exact_map:
        return exact_map[n], 100
    match = process.extractOne(
        query=n,
        choices=choices,
        scorer=fuzz.WRatio,          # changed from token_sort_ratio
        score_cutoff=score_cutoff
    )
    if not match:
        return None, 0
    matched_norm, score, _ = match
    return exact_map.get(matched_norm), score

def fill_fu_columns(
    xlsx_in: str,
    xlsx_out: str,
    portal_df: pd.DataFrame,
    sheet_name: str,
    db_name_col: str = "Name",
    score_cutoff: int = 95,
    snaps_col: str | None = None,     # set to "Snaps" if you have it, else leave None
    snaps_min: int = 100
):
    # --- diagnostics: portal duplicates by normalized name ---
    tmp = portal_df.copy()
    tmp["name_norm"] = tmp["name"].map(norm_name)
    dup_count = tmp.duplicated("name_norm", keep=False).sum()
    print(f"Portal rows: {len(tmp)} | duplicate normalized-name rows: {dup_count}")

    pdf, exact_map, choices = build_portal_lookup(portal_df)

    wb = load_workbook(xlsx_in)
    ws = wb[sheet_name]

    # headers
    header_row = 1
    headers = {}
    for col in range(1, ws.max_column + 1):
        v = ws.cell(row=header_row, column=col).value
        if v is not None:
            headers[str(v).strip()] = col

    required = [
        db_name_col,
        "247/On3 Position",
        "High School, City, State",
        "247 'Exp' (aka High School Class)",
        "H/S Stars",
        "H/S Rating",
        "H/S National Rank",
        "H/S Position Rank",
        "Transfer Year",
        "Transfer Origin",
        "Origin P4 / G5 / Non-FBS",
        "Transfer Destination",
        "Destination P4 / G5 / Non-FBS",
        "Transfer Stars",
        "Transfer Rating",
        "Transfer Overall Rank",
        "Transfer Position Rank",
    ]
    missing = [c for c in required if c not in headers]
    if missing:
        raise ValueError(f"Missing these headers in the sheet: {missing}")

    if snaps_col is not None and snaps_col not in headers:
        raise ValueError(f"snaps_col='{snaps_col}' not found in sheet headers")

    name_col_idx = headers[db_name_col]
    transfer_year_col = headers["Transfer Year"]

    # --- NEW: enforce one-to-one matching (no portal id reused) ---
    used_portal_ids = set()

    matched = 0
    unmatched = 0
    skipped_filled = 0
    skipped_snaps = 0
    skipped_reuse = 0
    below_cutoff = 0

    for r in range(header_row + 1, ws.max_row + 1):
        # --- NEW: GATE: only fill if Transfer Year is blank (prevents mass filling non-transfer rows) ---
        ty = ws.cell(row=r, column=transfer_year_col).value
        if ty not in (None, ""):
            skipped_filled += 1
            continue

        # --- optional GATE: snaps >= 100 ---
        if snaps_col is not None:
            sv = ws.cell(row=r, column=headers[snaps_col]).value
            try:
                s = float(sv) if sv not in (None, "") else 0.0
            except Exception:
                s = 0.0
            if s < snaps_min:
                skipped_snaps += 1
                continue

        name_raw = ws.cell(row=r, column=name_col_idx).value
        portal_idx, score = fuzzy_match_one(name_raw, exact_map, choices, score_cutoff=score_cutoff)

        if portal_idx is None:
            unmatched += 1
            continue
        if score < score_cutoff:
            below_cutoff += 1
            continue

        pid = int(pdf.loc[portal_idx, "id_247"])
        if pid in used_portal_ids:
            skipped_reuse += 1
            continue
        used_portal_ids.add(pid)

        prow = pdf.loc[portal_idx]
        vals = portal_to_fu_fields(prow)

        for k, v in vals.items():
            ws.cell(row=r, column=headers[k]).value = v

        matched += 1

    wb.save(xlsx_out)

    print(f"Saved: {xlsx_out}")
    print(f"Matched (written): {matched}")
    print(f"Unmatched: {unmatched}")
    print(f"Skipped (already had Transfer Year): {skipped_filled}")
    print(f"Skipped (snaps<{snaps_min}): {skipped_snaps}")
    print(f"Skipped (portal id reuse): {skipped_reuse}")
    print(f"Below cutoff (should be 0): {below_cutoff}")
    print(f"Unique portal ids used: {len(used_portal_ids)} (<= {len(portal_df)})")

# ---------- RUN ----------
xlsx_in  = "data/2024-2025 Player Database v2.xlsx"
xlsx_out = "data/2024-2025 Player Database v2_FILLED.xlsx"

fill_fu_columns(
    xlsx_in=xlsx_in,
    xlsx_out=xlsx_out,
    portal_df=portal_df_test,      # your 3008-row df already in memory
    sheet_name="2024-2025 Player Database CSV",          # change
    db_name_col="full_name",           # change if your header differs
    score_cutoff=95,
    snaps_col='pff_snaps'           # <- set to "Snaps" if your sheet has it and you want the 100+ snaps gate
)


Portal rows: 3008 | duplicate normalized-name rows: 189
Saved: data/2024-2025 Player Database v2_FILLED.xlsx
Matched (written): 2012
Unmatched: 8645
Skipped (already had Transfer Year): 0
Skipped (snaps<100): 20203
Skipped (portal id reuse): 913
Below cutoff (should be 0): 0
Unique portal ids used: 2012 (<= 3008)
